In [4]:
from train import parse_args, get_device, build_model, count_parameters, train_epoch, eval_epoch, save_checkpoint

In [5]:
args = parse_args()

torch.manual_seed(config.SEED)
np.random.seed(config.SEED)

device = get_device()
print(f"Device : {device}")

print("Loading and preprocessing data …")
train_loader, val_loader, _, _, _ = get_dataloaders(
    batch_size=args.batch_size, save_scalers=True
)

n_features = config.N_FEATURES
model = build_model(args.model, n_features).to(device)
print(f"Model  : {args.model.upper()} | Parameters: {count_parameters(model):,}")

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY
)
# OneCycleLR needs total_steps upfront; early stopping may shorten the run
total_steps = len(train_loader) * args.epochs
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=config.ONE_CYCLE_MAX_LR,
    total_steps=total_steps,
)

best_val_loss    = float("inf")
patience_counter = 0
history          = []

print(f"\nTraining up to {args.epochs} epochs  "
        f"(early-stop patience = {config.PATIENCE})\n")

for epoch in range(1, args.epochs + 1):
    train_loss = train_epoch(
        model, train_loader, optimizer, criterion, scheduler, device
    )
    val_loss = eval_epoch(model, val_loader, criterion, device)
    history.append((epoch, train_loss, val_loss))

    improved = val_loss < best_val_loss
    marker   = " [saved]" if improved else f" (patience {patience_counter + 1}/{config.PATIENCE})"
    print(f"Epoch {epoch:4d} | train MSE {train_loss:.6f} | val MSE {val_loss:.6f}{marker}")

    if improved:
        best_val_loss    = val_loss
        patience_counter = 0
        save_checkpoint(model, optimizer, epoch, val_loss, args.model, n_features)
    else:
        patience_counter += 1
        if patience_counter >= config.PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            break

# Save loss history
config.RESULTS_DIR.mkdir(exist_ok=True)
history_path = config.RESULTS_DIR / f"loss_history_{args.model}.csv"
with open(history_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_mse", "val_mse"])
    writer.writerows(history)

print(f"\nBest val MSE : {best_val_loss:.6f}")
print(f"Loss history : {history_path}")
print(f"Checkpoint   : {config.CHECKPOINT_DIR / f'best_model_{args.model}.pt'}")

usage: ipykernel_launcher.py [-h] --model {mlp,gru} [--epochs EPOCHS]
                             [--batch_size BATCH_SIZE]
ipykernel_launcher.py: error: the following arguments are required: --model


SystemExit: 2

c:\Users\maxma\miniforge3\envs\actuator_net_cpu\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
